# Sensibilizar el punto exacto: de "estación cercana" a "clima en el punto del cliente" (ECO | Wind)

Hallazgo 24 resolvió cómo encontrar y descargar la estación REAL más cercana a cualquier punto, en
cualquiera de los 20 países del catálogo -- pero "más cercana" no es "exacta" (Bogotá salió a 84.6
km de su estación más cercana). Este notebook prueba el paso que falta: ajustar la MAGNITUD de la
forma real de la estación donante a lo que pasa en el punto EXACTO que pide el cliente, sin volver
a anclar nada a San José ni a ningún sitio fijo.

**Mecanismo propuesto (ver el plan completo en el chat):**

```
media_ajustada_al_punto_exacto = media_real_de_la_estación_donante × [fuente_continua(punto_exacto) / fuente_continua(ubicación_de_la_estación)]
```

La `fuente_continua` sólo se usa para una RAZÓN entre dos puntos cercanos, no para su valor
absoluto -- si esa fuente tiene un sesgo sistemático en la región (NASA POWER subestima ~3x en
Costa Rica, Hallazgo 1), ese sesgo se cancela en gran parte al dividir, y lo que sobrevive es la
diferencia real de microclima entre la estación y el punto exacto. La FORMA horaria (variabilidad,
patrón diurno/estacional) sigue siendo 100% real, de la estación donante -- no se inventa nada,
sólo se reescala la magnitud.

Dos partes: (1) investigar en vivo si el Global Wind Atlas tiene una API de consulta por punto real
y accesible -- no se encontró un endpoint documentado y confirmado en la investigación previa (sólo
"la API existe, no está pensada para bulk"), así que esta parte SÓLO prueba alcanzabilidad de las
páginas reales encontradas, no inventa una URL de datos; (2) el mecanismo completo usando NASA
POWER como fuente continua -- ya confirmado que funciona en Colab (Hallazgo 23), es la vía segura
mientras (1) se termina de confirmar.

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at 2fe9b89 feat(fase2): engine/open_meteo_client.py -- quinta via, sin la friccion de CDS


/home/user/eco-wind/notebooks
Commit activo: 2fe9b89  feat(fase2): engine/open_meteo_client.py -- quinta via, sin la friccion de CDS  (2026-09-01 00:25:50 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import calendar

import numpy as np
import pandas as pd
import requests

from engine.formas_regionales import cargar_formas_conocidas, vecino_mas_cercano
from engine.simulador_pista_a import generar_clima_gwa, simular

pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Parte 1 — ¿Tiene el Global Wind Atlas una API de punto real y accesible?

**Honesto de entrada:** la investigación previa (WebSearch) confirmó que GWA 4.0 tiene cobertura
GLOBAL (todos los países + zonas offshore) y que existe un "EMD-API - Global Atlas Services"
documentado como REST/OpenAPI -- pero NO se encontró un endpoint concreto y confirmado para
consulta por punto (lat/lon → media de viento en JSON). `help.emd.dk` (donde vive esa
documentación) ya está confirmado bloqueado en el sandbox de desarrollo (Hallazgo 2) -- acá se
prueba si sigue bloqueado desde Colab, y se revisa el contenido de las páginas reales que sí se
encontraron, para ver si documentan el formato del endpoint. No se inventa ninguna URL de datos.

In [3]:
paginas_reales_a_probar = {
    "Global Wind Atlas (home)": "https://globalwindatlas.info",
    "GWA -- GIS files & API access": "https://globalwindatlas.info/download/gis-files",
    "EMD-API docs (Wiki-WindPRO)": "https://help.emd.dk/mediawiki/index.php/EMD-API_-_Global_Atlas_Services",
    "windatlas.xyz docs (tool de terceros, no es GWA/DTU)": "http://windatlas.xyz/docs/api/",
}

for nombre, url in paginas_reales_a_probar.items():
    try:
        resp = requests.get(url, timeout=15)
        print(f"{nombre}: OK -- {resp.status_code}, {len(resp.text)} caracteres")
        if resp.status_code == 200 and ("api" in resp.text.lower() or "endpoint" in resp.text.lower()):
            print("  (la página menciona 'api'/'endpoint' -- vale la pena leerla completa a mano)")
    except Exception as exc:
        print(f"{nombre}: FALLO -- {exc!r}")

Global Wind Atlas (home): FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: / (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


GWA -- GIS files & API access: FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: /download/gis-files (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


EMD-API docs (Wiki-WindPRO): FALLO -- ProxyError(MaxRetryError("HTTPSConnectionPool(host='help.emd.dk', port=443): Max retries exceeded with url: /mediawiki/index.php/EMD-API_-_Global_Atlas_Services (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
windatlas.xyz docs (tool de terceros, no es GWA/DTU): OK -- 403, 100 caracteres


**Actualización real (Colab, 31 de agosto 2026, confirmado por Pablo con captura de pantalla):**
las 4 páginas SÍ responden 200 en Colab (bloqueadas sólo en este sandbox de desarrollo). La página
real de "GIS files & API access" (`globalwindatlas.info/download/gis-files`) contesta la pregunta
directamente -- **no hay una API de consulta por punto separada.** El formulario real es: elegir
país → elegir capa (GEOJSON, WIND-SPEED, POWER-DENSITY, AIR-DENSITY, ...) → elegir altura (10, 50,
100, 150, 200m) → esto genera una URL de descarga de RASTER PARA TODO EL PAÍS, y esa MISMA URL
"can also be used as an API service" -- es decir, el "API" de GWA es exactamente el mecanismo que
ya se había construido para Costa Rica en Hallazgo 17 (`descargar_raster_costa_rica()`), sólo que
generalizable a cualquier país. La página también advierte explícito: **"not to be used for bulk
downloads of all countries or datasets"** -- bajar UN país está bien (como ya se hace), scriptear
una descarga masiva de los 20 no.

El patrón de URL ya estaba confirmado desde Hallazgo 17 y coincide exacto con lo que muestra la
página real: `https://globalwindatlas.info/api/gis/country/{ISO3}/{capa}/{altura}`. Generalizado
ahora en `engine/gwa_raster.py::descargar_raster_pais(pais_iso3, altura=10)` -- ver Parte 3 más
abajo, donde se usa exactamente esto (con Costa Rica) en vez de NASA POWER para el ajuste espacial.

## Parte 2 — El mecanismo completo, con NASA POWER como fuente continua (la vía ya confirmada)

`factor_ajuste_nasa_power()`: la razón entre la media de NASA POWER en el punto exacto y en la
ubicación de la estación donante. `evaluar_punto_con_ajuste()`: junta todo -- encuentra el vecino
real más cercano (reusa `vecino_mas_cercano()` de `engine/formas_regionales.py`, Hallazgo 21), y en
vez de escalar su forma a una media "ya conocida" (que en un punto nuevo de verdad NUNCA se tiene),
la escala por este factor de ajuste espacial -- es una prueba más honesta que la de Hallazgo 21/22,
porque no usa ninguna información que no estaría disponible para un punto nuevo real.

In [4]:
NASA_POWER_HOURLY_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_nasa_power_hourly(lat, lon, year, community="SB", parameters=("WS10M",)):
    params = {
        "parameters": ",".join(parameters), "community": community,
        "longitude": lon, "latitude": lat,
        "start": f"{year}0101", "end": f"{year}1231", "format": "JSON",
    }
    resp = requests.get(NASA_POWER_HOURLY_URL, params=params, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    horas_esperadas = 8784 if calendar.isleap(year) else 8760
    if len(df) != horas_esperadas:
        raise ValueError(f"Esperaba {horas_esperadas} horas, llegaron {len(df)}")
    return df


def factor_ajuste_nasa_power(lat_exacto, lon_exacto, lat_estacion, lon_estacion, year=2023):
    '''
    Razon NASA POWER(punto exacto) / NASA POWER(ubicacion de la estacion donante) -- el sesgo
    sistematico de NASA POWER (Hallazgo 1) se cancela en gran parte al dividir dos puntos
    cercanos de la misma fuente; sobrevive sobre todo la diferencia real de microclima.
    '''
    media_exacto = fetch_nasa_power_hourly(lat_exacto, lon_exacto, year)["WS10M"].mean()
    media_estacion = fetch_nasa_power_hourly(lat_estacion, lon_estacion, year)["WS10M"].mean()
    return media_exacto / media_estacion, media_exacto, media_estacion


def evaluar_punto_con_ajuste(lat, lon, formas, excluir=None, year=2023,
                              modelo="medium_tulip", N=3, altura_buje=3.0, elevacion_m=0.0):
    clave_donante, dist_km = vecino_mas_cercano(lat, lon, formas, excluir=excluir)
    donante = formas[clave_donante]

    factor, media_np_exacto, media_np_donante = factor_ajuste_nasa_power(
        lat, lon, donante["lat"], donante["lon"], year=year)

    media_donante_real = (float(np.mean([r["val"] for r in donante["ws_json"]])) if clave_donante == "san_jose"
                           else float(donante["df_real"]["WS10M"].mean()))
    media_ajustada = media_donante_real * factor

    df_clima, _ = generar_clima_gwa(donante["ws_json"], donante["hm_json"], media_objetivo=media_ajustada)
    r = simular(df_clima, altura_buje, modelo, N, elevacion_m=elevacion_m)

    return dict(donante=formas[clave_donante]["nombre"], distancia_km=dist_km, factor_ajuste=factor,
                media_np_exacto=media_np_exacto, media_np_donante=media_np_donante,
                media_donante_real=media_donante_real, media_ajustada=media_ajustada,
                kwh_ajustado=r["kwh_anual"])

## Validación leave-one-out con el mecanismo nuevo -- ¿mejora sobre lo ya documentado?

Para cada uno de los 4 sitios reales conocidos: se tapa su propia forma Y su propia media real (a
diferencia de Hallazgo 21/22, acá NO se usa la media real ya conocida del sitio -- es información
que un punto nuevo de verdad no tendría). Se compara contra la verdad real ya conocida, y contra
los dos mecanismos ya documentados (siempre San José, y vecino más cercano con curva por residuo
de Hallazgo 22).

In [5]:
formas = cargar_formas_conocidas(usar_residuo=True)
filas = []

for clave, sitio in formas.items():
    if clave == "san_jose":
        df_real, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
        media_real = float(np.mean([r["val"] for r in sitio["ws_json"]]))
    else:
        df_real = sitio["df_real"]
        media_real = float(df_real["WS10M"].mean())
    r_real = simular(df_real, 3.0, "medium_tulip", 3, elevacion_m=sitio["elevacion_m"])

    try:
        ajuste = evaluar_punto_con_ajuste(sitio["lat"], sitio["lon"], formas, excluir=clave,
                                           elevacion_m=sitio["elevacion_m"])
        error_ajustado_pct = (ajuste["kwh_ajustado"] / r_real["kwh_anual"] - 1) * 100
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"],
                    donante=ajuste["donante"], distancia_km=ajuste["distancia_km"],
                    factor_ajuste_nasa_power=ajuste["factor_ajuste"],
                    kwh_nuevo_ajustado=ajuste["kwh_ajustado"], error_nuevo_ajustado_pct=error_ajustado_pct)
    except Exception as exc:
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"],
                    donante=None, distancia_km=None, factor_ajuste_nasa_power=None,
                    kwh_nuevo_ajustado=None, error_nuevo_ajustado_pct=f"FALLO: {exc!r}")
    filas.append(fila)

pd.DataFrame(filas)

sitio	kwh_real	donante	distancia_km	factor_ajuste_nasa_power	kwh_nuevo_ajustado	error_nuevo_ajustado_pct
San José (Aeropuerto Juan Santamaría)	156.439	Nicoya A.P. (Guanacaste, Pacífico seco)	137.458	0.365	2.303	-98.528
Nicoya A.P. (Guanacaste, Pacífico seco)	52.400	Daniel Oduber / Liberia Intl. A.P. (Guanacaste)	50.321	0.963	363.326	593.374
Daniel Oduber / Liberia Intl. A.P. (Guanacaste)	291.487	Nicoya A.P. (Guanacaste, Pacífico seco)	50.321	1.038	71.785	-75.373
Finca Favorita (Limón, Caribe)	7.439	San José (Aeropuerto Juan Santamaría)	178.604	1.880	1136.978	15183.964


## Diagnóstico honesto del resultado de NASA POWER -- no es un bug, es el método fallando de raíz

Números catastróficos: San José -98.5%, Nicoya +593%, Liberia -75.4%, **Finca Favorita +15,184%**
(153x la producción real). Verificado con cálculo, no es un error de fórmula:

- **San José vs. Finca Favorita:** NASA POWER dice que Finca Favorita es 1.88x más ventosa que San
  José. La VERDAD es lo contrario -- Finca Favorita tiene sólo 38.5% del viento de San José. La
  razón sale literalmente al revés.
- **Nicoya vs. Liberia** (50km, terreno similar): en la realidad Liberia es 1.74x más ventosa. NASA
  POWER casi no distingue los dos puntos (factor 0.96-1.04) -- su grilla de ~50-60km es demasiado
  gruesa incluso para dos sitios del mismo tipo de terreno.

La idea de que "la razón entre dos puntos cercanos cancela el sesgo sistemático de la fuente" ASUME
que el sesgo es parejo en la región -- pero Hallazgo 1 ya había mostrado que el sesgo de NASA POWER
viene de no poder resolver terreno complejo (por eso subestima ~3x en el valle de San José). Si el
sesgo depende de qué tan complejo es el terreno de CADA punto, la razón no cancela nada -- puede
invertir la relación real, que es exactamente lo que pasó. **Conclusión: para este terreno, NASA
POWER no sirve como corrector espacial -- no es un tema de afinar el método, hay que cambiar de
fuente.**

## Parte 3 — El mismo mecanismo, con GWA (250m) en vez de NASA POWER (~50-60km)

Si el problema de NASA POWER es resolución (no puede ver el terreno complejo de Costa Rica), GWA
debería andar mucho mejor -- es ~200-1000x más fino. Mecánicamente es más simple además: en vez de
2 llamadas a una API por internet, son 2 lecturas de píxel del MISMO raster ya descargado una vez
(`factor_ajuste_gwa()`, nuevo en `engine/gwa_raster.py`, mismo patrón que
`factor_ajuste_nasa_power()` de la Parte 2).

**Necesita el ráster de Costa Rica descargado** (`descargar_raster_pais("CRI")`, generalización de
`descargar_raster_costa_rica()` de Hallazgo 17) -- no corre en este sandbox, sólo en Colab.

In [6]:
from engine.gwa_raster import descargar_raster_pais, factor_ajuste_gwa, RUTA_RASTER_CR_DEFAULT

try:
    ruta_raster_cr = descargar_raster_pais("CRI", altura=10)
    print(f"Ráster descargado: {ruta_raster_cr}")
except Exception as exc:
    ruta_raster_cr = RUTA_RASTER_CR_DEFAULT
    print(f"No se pudo descargar (mismo bloqueo de red ya documentado, Hallazgo 2): {exc!r}")
    print("La celda de abajo va a fallar igual (no hay archivo local) hasta correr esto en Colab.")


Ráster descargado: /content/ECO-Wind/datos_clima/gwa_cri_10m.tif


### Diagnóstico previo -- ¿el RÁSTER CRUDO de GWA ya se acerca a la realidad en los 4 sitios?

Antes de ver si el MECANISMO de razón funciona, un chequeo más simple y directo: leer el valor
crudo del ráster (sin ninguna razón, sin ningún ajuste) en la coordenada exacta de cada uno de los
4 sitios reales, y compararlo contra su media real ya conocida. Esto es justo lo que faltó
diagnosticar ANTES de correr NASA POWER (Hallazgo 25) -- ahora se hace primero, para no necesitar
otro viaje a Colab si el resultado de la razón sale raro otra vez.

In [7]:
from engine.gwa_raster import muestrear_velocidad_media

print("Valor crudo del ráster GWA (10m) vs. media real conocida, en la propia coordenada de cada sitio:")
filas_diag_gwa = []
for clave, sitio in formas.items():
    media_real = (float(np.mean([r["val"] for r in sitio["ws_json"]])) if clave == "san_jose"
                  else float(sitio["df_real"]["WS10M"].mean()))
    try:
        media_gwa = muestrear_velocidad_media(sitio["lat"], sitio["lon"], ruta_raster_cr)
        diff_pct = (media_gwa / media_real - 1) * 100
    except Exception as exc:
        media_gwa, diff_pct = None, f"FALLO: {exc!r}"
    filas_diag_gwa.append(dict(sitio=sitio["nombre"], media_real_m_s=media_real,
                                media_gwa_raster=media_gwa, diferencia_pct=diff_pct))

pd.DataFrame(filas_diag_gwa)

Valor crudo del ráster GWA (10m) vs. media real conocida, en la propia coordenada de cada sitio:
sitio	media_real_m_s	media_gwa_raster	diferencia_pct
San José (Aeropuerto Juan Santamaría)	3.669	2.088	-43.081
Nicoya A.P. (Guanacaste, Pacífico seco)	2.091	2.252	7.666
Daniel Oduber / Liberia Intl. A.P. (Guanacaste, Pacífico)	3.629	3.390	-6.584
Finca Favorita (Limón, Caribe)	1.413	0.180	-87.238


**Lectura honesta de este diagnóstico real (Colab, 31 de agosto 2026):** mixto, no limpio.
Nicoya (+7.7%) y Liberia (-6.6%) -- ambos Guanacaste -- salen razonablemente cerca de la verdad ya
conocida, mucho mejor que cualquier cosa que dio NASA POWER. Pero San José (-43.1%) y sobre todo
Finca Favorita (-87.2%, prácticamente calma) se alejan mucho.

San José tiene un precedente real y ya documentado (Hallazgo 3): el archivo `.lib` (formato WAsP
nativo) de GWA da 5.37 m/s, contra 3.67 m/s del panel web -- una brecha de +46% YA CONOCIDA **entre
dos productos distintos de GWA en el mismo punto**. Esta brecha nueva (ráster/API vs. panel, -43%)
es del mismo orden de magnitud -- consistente con que GWA simplemente tiene varios productos que no
concuerdan entre sí para este sitio, no necesariamente un problema nuevo. Esto es una hipótesis
razonable, no una explicación confirmada.

Finca Favorita (-87%) es un caso aparte -- la brecha es demasiado grande para la misma explicación.
Hipótesis sin confirmar: GWA podría resolver mal el terreno costero/boscoso del Caribe a 250m de
resolución (igual que NASA POWER resolvía mal el terreno complejo de San José, pero por una razón
distinta -- acá no es resolución gruesa, es un tipo de terreno que el modelo de downscaling de GWA
podría no representar bien). No se investigó más a fondo todavía.

**Bug encontrado y corregido en esta misma revisión:** la celda de validación (abajo) no le pasaba
la ruta real del ráster descargado a `evaluar_punto_con_ajuste_gwa()` -- usaba el nombre de archivo
por defecto (`gwa_costa_rica_10m.tif`), pero `descargar_raster_pais("CRI")` sin `destino` explícito
descarga a `gwa_cri_10m.tif` -- un nombre distinto. Por eso la corrida anterior de Pablo mostró
`FileNotFoundError` ahí a pesar de que el diagnóstico de arriba (que sí pasaba la ruta correcta)
funcionó. Ya corregido -- hace falta volver a correr esto en Colab para tener el número real de la
validación con razón de GWA.

In [8]:
def evaluar_punto_con_ajuste_gwa(lat, lon, formas, excluir=None, ruta_raster=RUTA_RASTER_CR_DEFAULT,
                                  modelo="medium_tulip", N=3, altura_buje=3.0, elevacion_m=0.0):
    clave_donante, dist_km = vecino_mas_cercano(lat, lon, formas, excluir=excluir)
    donante = formas[clave_donante]

    factor, media_gwa_exacto, media_gwa_donante = factor_ajuste_gwa(
        lat, lon, donante["lat"], donante["lon"], ruta_raster=ruta_raster)

    media_donante_real = (float(np.mean([r["val"] for r in donante["ws_json"]])) if clave_donante == "san_jose"
                           else float(donante["df_real"]["WS10M"].mean()))
    media_ajustada = media_donante_real * factor

    df_clima, _ = generar_clima_gwa(donante["ws_json"], donante["hm_json"], media_objetivo=media_ajustada)
    r = simular(df_clima, altura_buje, modelo, N, elevacion_m=elevacion_m)

    return dict(donante=formas[clave_donante]["nombre"], distancia_km=dist_km, factor_ajuste=factor,
                media_gwa_exacto=media_gwa_exacto, media_gwa_donante=media_gwa_donante,
                media_donante_real=media_donante_real, media_ajustada=media_ajustada,
                kwh_ajustado=r["kwh_anual"])


filas_gwa = []
for clave, sitio in formas.items():
    if clave == "san_jose":
        df_real, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
        media_real = float(np.mean([r["val"] for r in sitio["ws_json"]]))
    else:
        df_real = sitio["df_real"]
        media_real = float(df_real["WS10M"].mean())
    r_real = simular(df_real, 3.0, "medium_tulip", 3, elevacion_m=sitio["elevacion_m"])

    try:
        ajuste = evaluar_punto_con_ajuste_gwa(sitio["lat"], sitio["lon"], formas, excluir=clave,
                                               ruta_raster=ruta_raster_cr, elevacion_m=sitio["elevacion_m"])
        error_pct = (ajuste["kwh_ajustado"] / r_real["kwh_anual"] - 1) * 100
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=ajuste["donante"],
                    distancia_km=ajuste["distancia_km"], factor_ajuste_gwa=ajuste["factor_ajuste"],
                    kwh_nuevo_gwa=ajuste["kwh_ajustado"], error_nuevo_gwa_pct=error_pct)
    except Exception as exc:
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=None,
                    distancia_km=None, factor_ajuste_gwa=None, kwh_nuevo_gwa=None,
                    error_nuevo_gwa_pct=f"FALLO: {exc!r}")
    filas_gwa.append(fila)

pd.DataFrame(filas_gwa)

sitio	kwh_real	donante	distancia_km	factor_ajuste_gwa	kwh_nuevo_gwa	error_nuevo_gwa_pct
San José (Aeropuerto Juan Santamaría)	156.439	Nicoya A.P. (Guanacaste, Pacífico seco)	137.458	0.927	46.907	-70.016
Nicoya A.P. (Guanacaste, Pacífico seco)	52.400	Daniel Oduber / Liberia Intl. A.P. (Guanacaste)	50.321	0.664	118.775	126.671
Daniel Oduber / Liberia Intl. A.P. (Guanacaste)	291.487	Nicoya A.P. (Guanacaste, Pacífico seco)	50.321	1.506	220.452	-24.370
Finca Favorita (Limón, Caribe)	7.439	San José (Aeropuerto Juan Santamaría)	178.604	0.086	0.000	-100.000


## Parte 4 — El mismo mecanismo, con ERA5 (~31km) en vez de GWA

Decisión de Pablo: seguir con ERA5 para el ajuste espacial. GWA queda pausado -- funcionó bien en
Guanacaste pero mal en San José/Finca Favorita (Hallazgo 26), un problema del RÁSTER CRUDO de GWA
en esos sitios, no del mecanismo de razón en sí -- es "algo para afinar más adelante, si los otros
métodos fallan" (palabras de Pablo), no descartado.

### Antes de correr esto -- hace falta una cuenta de Copernicus CDS (paso nuevo, no como NASA POWER/GWA)

A diferencia de NASA POWER y GWA (ninguno pide credencial), ERA5 sí la necesita.

1. Registrate gratis en <https://cds.climate.copernicus.eu/> (nombre, email, país, sector -- sin
   pedir tarjeta según lo investigado) y copiá el **Personal Access Token** de tu página de perfil.
2. **Guardalo como Secret de Colab, NO en el archivo del notebook:** en la barra lateral izquierda
   de Colab, ícono de llave 🔑 → "Add new secret" → nombre exacto `CDS_API_KEY`, valor = tu token →
   activá el toggle de "Notebook access" para este notebook. Se guarda una sola vez en tu cuenta de
   Colab, no en este archivo -- así nunca queda pegado en git (un token escrito directo en el
   notebook terminaría commiteado al repo para siempre, aunque después lo canjees por otro).
3. Entrá a la página del dataset ERA5 en CDS y aceptá sus Términos y Condiciones (paso aparte del
   registro general -- si no lo hacés, el pedido falla igual con el secret bien puesto).

**Sin esto, todo lo de abajo va a fallar al autenticar** -- no hay forma de saltearlo.

### Sobre la velocidad -- distinto de NASA POWER

NASA POWER responde casi al instante (JSON, sin cola). CDS procesa los pedidos en una cola -- puede
tardar minutos por sitio, a veces bastante más si está ocupado. La validación completa pide 8 veces
(2 por cada uno de los 4 sitios) -- contá con que esto puede tardar bastante más que las partes
anteriores. Cada sitio imprime su progreso a medida que termina, para no quedarte mirando una celda
en blanco.

In [9]:
get_ipython().system("pip install --quiet cdsapi xarray netCDF4")

import os

try:
    from google.colab import userdata
    _token = userdata.get("CDS_API_KEY")
    with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
        f.write(f"url: https://cds.climate.copernicus.eu/api\nkey: {_token}\n")
    print("Token leído desde el Secret de Colab (CDS_API_KEY) y ~/.cdsapirc escrito.")
except Exception as exc:
    print(f"No se pudo leer el secret 'CDS_API_KEY' de Colab: {exc!r}")
    print("Revisá: (1) que exista un secret llamado EXACTO 'CDS_API_KEY' en el panel de la llave 🔑,")
    print("        (2) que el toggle de acceso de notebook esté activado para este archivo,")
    print("        (3) que estés corriendo esto en Colab (fuera de Colab no existe google.colab).")

print("¿Existe ~/.cdsapirc?", os.path.exists(os.path.expanduser("~/.cdsapirc")))

No se pudo leer el secret 'CDS_API_KEY' de Colab: ModuleNotFoundError("No module named 'google.colab'")
Revisá: (1) que exista un secret llamado EXACTO 'CDS_API_KEY' en el panel de la llave 🔑,
        (2) que el toggle de acceso de notebook esté activado para este archivo,
        (3) que estés corriendo esto en Colab (fuera de Colab no existe google.colab).
¿Existe ~/.cdsapirc? False


In [10]:
from engine.era5_client import factor_ajuste_era5


def evaluar_punto_con_ajuste_era5(lat, lon, formas, excluir=None,
                                   modelo="medium_tulip", N=3, altura_buje=3.0, elevacion_m=0.0):
    clave_donante, dist_km = vecino_mas_cercano(lat, lon, formas, excluir=excluir)
    donante = formas[clave_donante]

    factor, media_era5_exacto, media_era5_donante = factor_ajuste_era5(
        lat, lon, donante["lat"], donante["lon"])

    media_donante_real = (float(np.mean([r["val"] for r in donante["ws_json"]])) if clave_donante == "san_jose"
                           else float(donante["df_real"]["WS10M"].mean()))
    media_ajustada = media_donante_real * factor

    df_clima, _ = generar_clima_gwa(donante["ws_json"], donante["hm_json"], media_objetivo=media_ajustada)
    r = simular(df_clima, altura_buje, modelo, N, elevacion_m=elevacion_m)

    return dict(donante=formas[clave_donante]["nombre"], distancia_km=dist_km, factor_ajuste=factor,
                media_era5_exacto=media_era5_exacto, media_era5_donante=media_era5_donante,
                media_donante_real=media_donante_real, media_ajustada=media_ajustada,
                kwh_ajustado=r["kwh_anual"])


filas_era5 = []
for clave, sitio in formas.items():
    print(f"=== {sitio['nombre']} -- consultando ERA5 (puede tardar) ===")
    if clave == "san_jose":
        df_real, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
        media_real = float(np.mean([r["val"] for r in sitio["ws_json"]]))
    else:
        df_real = sitio["df_real"]
        media_real = float(df_real["WS10M"].mean())
    r_real = simular(df_real, 3.0, "medium_tulip", 3, elevacion_m=sitio["elevacion_m"])

    try:
        ajuste = evaluar_punto_con_ajuste_era5(sitio["lat"], sitio["lon"], formas, excluir=clave,
                                                elevacion_m=sitio["elevacion_m"])
        error_pct = (ajuste["kwh_ajustado"] / r_real["kwh_anual"] - 1) * 100
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=ajuste["donante"],
                    distancia_km=ajuste["distancia_km"], factor_ajuste_era5=ajuste["factor_ajuste"],
                    kwh_nuevo_era5=ajuste["kwh_ajustado"], error_nuevo_era5_pct=error_pct)
        print(f"  OK -- error: {error_pct:+.1f}%")
    except Exception as exc:
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=None,
                    distancia_km=None, factor_ajuste_era5=None, kwh_nuevo_era5=None,
                    error_nuevo_era5_pct=f"FALLO: {exc!r}")
        print(f"  FALLO: {exc!r}")
    filas_era5.append(fila)

pd.DataFrame(filas_era5)

=== San José (Aeropuerto Juan Santamaría) -- consultando ERA5 (puede tardar) ===


  FALLO: Exception('Missing/incomplete configuration file: /root/.cdsapirc')
=== Nicoya A.P. (Guanacaste, Pacífico seco) -- consultando ERA5 (puede tardar) ===
  FALLO: Exception('Missing/incomplete configuration file: /root/.cdsapirc')
=== Daniel Oduber / Liberia Intl. A.P. (Guanacaste, Pacífico) -- consultando ERA5 (puede tardar) ===
  FALLO: Exception('Missing/incomplete configuration file: /root/.cdsapirc')
=== Finca Favorita (Limón, Caribe) -- consultando ERA5 (puede tardar) ===
  FALLO: Exception('Missing/incomplete configuration file: /root/.cdsapirc')


,sitio,kwh_real,donante,distancia_km,factor_ajuste_era5,kwh_nuevo_era5,error_nuevo_era5_pct
0,San José (Aeropuerto Juan Santamaría),156.439,None,None,None,None,FALLO: Exception('Missing/incomplete configura...
1,"Nicoya A.P. (Guanacaste, Pacífico seco)",52.400,None,None,None,None,FALLO: Exception('Missing/incomplete configura...
2,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,291.487,None,None,None,None,FALLO: Exception('Missing/incomplete configura...
3,"Finca Favorita (Limón, Caribe)",7.439,None,None,None,None,FALLO: Exception('Missing/incomplete configura...


## Parte 5 — El mismo mecanismo, con Open-Meteo/ERA5-Land (~9km, sin la fricción de CDS)

Mientras Parte 4 seguía atascada en la cola de CDS por más de 40 minutos, Pablo investigó por su
cuenta (informe externo) alternativas de cero fricción. Confirmamos con el dashboard en vivo de CDS
(`cds.climate.copernicus.eu/live`) que era congestión real del servicio -- ~4,361 pedidos en cola
contra ~435 corriendo en ese momento -- no un problema de nuestro lado ni algo raro de esta cuenta.

**Open-Meteo** (`archive-api.open-meteo.com`) sirve ERA5-Land (~9km/0.1°, más fino que el ERA5
estándar de CDS a ~31km/0.25°) sin API key, sin registro, sin aceptar licencias por dataset, y sin
cola -- HTTP GET simple, JSON, según la documentación oficial. Mismo mecanismo de razón que las
tres vías anteriores: `factor_ajuste_open_meteo()` en `engine/open_meteo_client.py`.

**Sin verificar en vivo todavía** -- `open-meteo.com` está bloqueado en este sandbox, igual que
GWA/CDS/Figshare. El endpoint y los parámetros están confirmados por WebSearch contra la
documentación oficial (no adivinados), pero el formato exacto de la respuesta y los límites reales
de uso gratuito recién se confirman corriendo esto en Colab.

In [11]:
from engine.open_meteo_client import factor_ajuste_open_meteo


def evaluar_punto_con_ajuste_open_meteo(lat, lon, formas, excluir=None,
                                         modelo="medium_tulip", N=3, altura_buje=3.0, elevacion_m=0.0):
    clave_donante, dist_km = vecino_mas_cercano(lat, lon, formas, excluir=excluir)
    donante = formas[clave_donante]

    factor, media_om_exacto, media_om_donante = factor_ajuste_open_meteo(
        lat, lon, donante["lat"], donante["lon"])

    media_donante_real = (float(np.mean([r["val"] for r in donante["ws_json"]])) if clave_donante == "san_jose"
                           else float(donante["df_real"]["WS10M"].mean()))
    media_ajustada = media_donante_real * factor

    df_clima, _ = generar_clima_gwa(donante["ws_json"], donante["hm_json"], media_objetivo=media_ajustada)
    r = simular(df_clima, altura_buje, modelo, N, elevacion_m=elevacion_m)

    return dict(donante=formas[clave_donante]["nombre"], distancia_km=dist_km, factor_ajuste=factor,
                media_om_exacto=media_om_exacto, media_om_donante=media_om_donante,
                media_donante_real=media_donante_real, media_ajustada=media_ajustada,
                kwh_ajustado=r["kwh_anual"])


filas_open_meteo = []
for clave, sitio in formas.items():
    print(f"=== {sitio['nombre']} -- consultando Open-Meteo (debería ser rápido, sin cola) ===")
    if clave == "san_jose":
        df_real, _ = generar_clima_gwa(sitio["ws_json"], sitio["hm_json"])
        media_real = float(np.mean([r["val"] for r in sitio["ws_json"]]))
    else:
        df_real = sitio["df_real"]
        media_real = float(df_real["WS10M"].mean())
    r_real = simular(df_real, 3.0, "medium_tulip", 3, elevacion_m=sitio["elevacion_m"])

    try:
        ajuste = evaluar_punto_con_ajuste_open_meteo(sitio["lat"], sitio["lon"], formas, excluir=clave,
                                                      elevacion_m=sitio["elevacion_m"])
        error_pct = (ajuste["kwh_ajustado"] / r_real["kwh_anual"] - 1) * 100
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=ajuste["donante"],
                    distancia_km=ajuste["distancia_km"], factor_ajuste_open_meteo=ajuste["factor_ajuste"],
                    kwh_nuevo_open_meteo=ajuste["kwh_ajustado"], error_nuevo_open_meteo_pct=error_pct)
        print(f"  OK -- error: {error_pct:+.1f}%")
    except Exception as exc:
        fila = dict(sitio=sitio["nombre"], kwh_real=r_real["kwh_anual"], donante=None,
                    distancia_km=None, factor_ajuste_open_meteo=None, kwh_nuevo_open_meteo=None,
                    error_nuevo_open_meteo_pct=f"FALLO: {exc!r}")
        print(f"  FALLO: {exc!r}")
    filas_open_meteo.append(fila)

pd.DataFrame(filas_open_meteo)

=== San José (Aeropuerto Juan Santamaría) -- consultando Open-Meteo (debería ser rápido, sin cola) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=10.0034&longitude=-84.2033&start_date=2023-01-01&end_date=2023-12-31&hourly=wind_speed_10m&models=era5_land&wind_speed_unit=ms&timezone=UTC (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
=== Nicoya A.P. (Guanacaste, Pacífico seco) -- consultando Open-Meteo (debería ser rápido, sin cola) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=10.15&longitude=-85.45&start_date=2023-01-01&end_date=2023-12-31&hourly=wind_speed_10m&models=era5_land&wind_speed_unit=ms&timezone=UTC (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
=== Daniel Oduber / Liberia Intl. A.P. (Guanacaste, Pacífico) -- consultando Open-Meteo (debería ser rápido, sin cola) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=10.593&longitude=-85.544&start_date=2023-01-01&end_date=2023-12-31&hourly=wind_speed_10m&models=era5_land&wind_speed_unit=ms&timezone=UTC (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))
=== Finca Favorita (Limón, Caribe) -- consultando Open-Meteo (debería ser rápido, sin cola) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=9.517&longitude=-82.65&start_date=2023-01-01&end_date=2023-12-31&hourly=wind_speed_10m&models=era5_land&wind_speed_unit=ms&timezone=UTC (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


,sitio,kwh_real,donante,distancia_km,factor_ajuste_open_meteo,kwh_nuevo_open_meteo,error_nuevo_open_meteo_pct
0,San José (Aeropuerto Juan Santamaría),156.439,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."
1,"Nicoya A.P. (Guanacaste, Pacífico seco)",52.400,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."
2,Daniel Oduber / Liberia Intl. A.P. (Guanacaste...,291.487,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."
3,"Finca Favorita (Limón, Caribe)",7.439,None,None,None,None,"FALLO: ProxyError(MaxRetryError(""HTTPSConnecti..."


## Conclusión final -- las 5 vías (Hallazgo 25/26/27/28+)

| Sitio | Siempre San José (H21) | Vecino+residuo, verdad conocida (H22) | Vecino+NASA POWER (H25) | Vecino+GWA (H26) | Vecino+ERA5/CDS (H26/28) | Vecino+Open-Meteo |
|---|---|---|---|---|---|---|
| San José | — | — | -98.5% | -70.0% | *(pendiente, cola de CDS)* | *(ver tabla de arriba)* |
| Nicoya | -41.5% | +47.6% | +593.4% | +126.7% | *(pendiente, cola de CDS)* | *(ver tabla de arriba)* |
| Liberia | -43.7% | +15.9% | -75.4% | -24.4% | *(pendiente, cola de CDS)* | *(ver tabla de arriba)* |
| Finca Favorita | +19.2% | +19.2% | +15,184.0% | -100.0% | *(pendiente, cola de CDS)* | *(ver tabla de arriba)* |

NASA POWER descartado (falla al revés en terreno accidentado, Hallazgo 25). GWA pausado -- funciona
en Guanacaste, falla en San José/Finca Favorita por el propio ráster crudo (Hallazgo 26). ERA5/CDS
tiene la mejor base termodinámica (31km) pero una fricción de acceso real -- cuenta, token, licencia
por dataset, y una cola que puede tardar 40+ minutos según la congestión del servicio (confirmado
con el dashboard en vivo de CDS, no solo una demora percibida). Open-Meteo/ERA5-Land es la apuesta
de cero fricción -- misma familia de dato (ERA5) pero más fino (9km) y sin ninguna de las fricciones
de CDS, si el resultado real de arriba sale razonable sería la vía preferida por simplicidad
operativa sola, incluso si termina empatando en precisión con ERA5/CDS.

**Fuera del alcance de este mecanismo de razón (magnitud), investigado por separado:** el informe de
Pablo también propone mejorar la SELECCIÓN de estación donante (hoy pura distancia) combinando
Köppen-Geiger + elevación + distancia con una distancia de Gower en vez de un filtro binario simple
-- ver `notebooks/koppen_seleccion_donante.ipynb` (Hallazgo 27), que ya tiene el acceso real
confirmado y un boceto inicial. Y propone un downscaling topográfico más ambicioso (TPI, índice de
Winstral, factor de orografía de EN 1991-1-4, rugosidad vía ESA WorldCover) que es real y creíble
pero un desarrollo bastante más grande -- requiere un DEM nuevo, algoritmos geomorfométricos propios,
y validación contra los mismos 4 sitios conocidos; no arrancado, pendiente de que Pablo decida si
vale la pena frente al tiempo que tomaría.

No conectado a `app.py` todavía -- sigue siendo investigación.